<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embedddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  !pip install gensim
  from gensim.models import Word2Vec

### Datos
Utilizaremos como dataset canciones de bandas de habla inglesa.

In [10]:
# Descargar la carpeta de dataset
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl -L https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget -O songs_dataset.zip https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

El dataset ya se encuentra descargado


In [11]:
# Posibles bandas
os.listdir("./songs_dataset/")

['prince.txt',
 'dickinson.txt',
 'notorious-big.txt',
 'beatles.txt',
 'bob-dylan.txt',
 'bjork.txt',
 'johnny-cash.txt',
 'disney.txt',
 'janisjoplin.txt',
 'kanye.txt',
 'bob-marley.txt',
 'leonard-cohen.txt',
 'ludacris.txt',
 'adele.txt',
 'alicia-keys.txt',
 'joni-mitchell.txt',
 'amy-winehouse.txt',
 'lorde.txt',
 'rihanna.txt',
 'Kanye_West.txt',
 'nirvana.txt',
 'cake.txt',
 'bieber.txt',
 'notorious_big.txt',
 'missy-elliott.txt',
 'dolly-parton.txt',
 'jimi-hendrix.txt',
 'michael-jackson.txt',
 'al-green.txt',
 'lil-wayne.txt',
 'lady-gaga.txt',
 'lin-manuel-miranda.txt',
 'nursery_rhymes.txt',
 'dj-khaled.txt',
 'radiohead.txt',
 'patti-smith.txt',
 'blink-182.txt',
 'Lil_Wayne.txt',
 'dr-seuss.txt',
 'r-kelly.txt',
 'drake.txt',
 'britney-spears.txt',
 'bruce-springsteen.txt',
 'nicki-minaj.txt',
 'kanye-west.txt',
 'paul-simon.txt',
 'nickelback.txt',
 'eminem.txt',
 'bruno-mars.txt']

In [12]:
# Armar el dataset utilizando salto de línea para separar las oraciones/docs
df = pd.read_csv('songs_dataset/beatles.txt', sep='/n', header=None)
df.head()

/var/folders/h5/858gwy311wl8z7xr0xvpb78r0000gn/T/ipykernel_28663/3849064916.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv('songs_dataset/beatles.txt', sep='/n', header=None)


,0
0,"Yesterday, all my troubles seemed so far away"
1,Now it looks as though they're here to stay
2,"Oh, I believe in yesterday Suddenly, I'm not h..."
3,There's a shadow hanging over me.
4,"Oh, yesterday came suddenly Why she had to go ..."


In [5]:
print("Cantidad de documentos:", df.shape[0])

Cantidad de documentos: 1846


### 1 - Preprocesamiento

In [13]:
from gensim.utils import simple_preprocess

# Tokenizar cada línea para evitar dependencias pesadas de TensorFlow (solucion a error en MacOS)
sentence_tokens = [
    simple_preprocess(text, deacc=True)
    for text in df[0].dropna().astype(str).tolist()
]

In [14]:
# Demos un vistazo
sentence_tokens[:2]

[['yesterday', 'all', 'my', 'troubles', 'seemed', 'so', 'far', 'away'],
 ['now', 'it', 'looks', 'as', 'though', 'they', 're', 'here', 'to', 'stay']]

### 2 - Crear los vectores (word2vec)

In [15]:
from gensim.models.callbacks import CallbackAny2Vec
# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [16]:
# Crearmos el modelo generador de vectores
# En este caso utilizaremos la estructura modelo Skipgram
w2v_model = Word2Vec(min_count=5,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=2,       # cant de palabras antes y desp de la predicha
                     vector_size=300,       # dimensionalidad de los vectores
                     negative=20,    # cantidad de negative samples... 0 es no se usa
                     workers=1,      # si tienen más cores pueden cambiar este valor
                     sg=1)           # modelo 0:CBOW  1:skipgram

In [10]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [11]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 1846


In [12]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 431


### 3 - Entrenar embeddings

In [13]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=20,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

Loss after epoch 0: 105354.140625
Loss after epoch 1: 61648.875
Loss after epoch 2: 60364.875
Loss after epoch 3: 60546.203125
Loss after epoch 4: 59536.4375
Loss after epoch 5: 58869.53125
Loss after epoch 6: 58715.28125
Loss after epoch 7: 57317.34375
Loss after epoch 8: 58412.1875
Loss after epoch 9: 55974.5
Loss after epoch 10: 54930.6875
Loss after epoch 11: 54311.6875
Loss after epoch 12: 54136.125
Loss after epoch 13: 52530.5
Loss after epoch 14: 51874.0625
Loss after epoch 15: 51031.25
Loss after epoch 16: 50610.4375
Loss after epoch 17: 49104.125
Loss after epoch 18: 45554.875
Loss after epoch 19: 45019.125


(147673, 271740)

### 4 - Ensayar

In [ ]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["darling"], topn=10)

[('pretty', 0.8954247832298279),
 ('sleep', 0.8665655851364136),
 ('help', 0.8439376354217529),
 ('cry', 0.8351269960403442),
 ('not', 0.8309612274169922),
 ('try', 0.8276943564414978),
 ('peace', 0.8144856691360474),
 ('little', 0.8140572309494019),
 ('twist', 0.8123919367790222),
 ('seems', 0.8079564571380615)]

In [ ]:
# Palabras que MENOS se relacionan con...:
w2v_model.wv.most_similar(negative=["love"], topn=10)

[('shake', -0.22873197495937347),
 ('four', -0.2330218255519867),
 ('five', -0.23746445775032043),
 ('six', -0.23784494400024414),
 ('bang', -0.24832050502300262),
 ('our', -0.25539135932922363),
 ('day', -0.2689811885356903),
 ('going', -0.2692062556743622),
 ('here', -0.26990723609924316),
 ('three', -0.2838989198207855)]

In [ ]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["four"], topn=10)

[('five', 0.9813723564147949),
 ('three', 0.9745770692825317),
 ('six', 0.9710808992385864),
 ('seven', 0.9584357738494873),
 ('two', 0.9517216682434082),
 ('sixty', 0.8990395665168762),
 ('one', 0.7951181530952454),
 ('crying', 0.7946289777755737),
 ('us', 0.7740051746368408),
 ("i'm", 0.7508383393287659)]

In [ ]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["money"], topn=5)

[("can't", 0.9434017539024353),
 ('buy', 0.9396998882293701),
 ('much', 0.9033146500587463),
 ('just', 0.8509082198143005),
 ('hide', 0.835538387298584)]

In [ ]:
# Ensayar con una palabra que no está en el vocabulario:
w2v_model.wv.most_similar(negative=["diedaa"])

KeyError: "Key 'diedaa' not present in vocabulary"

In [ ]:
# el método `get_vector` permite obtener los vectores:
vector_love = w2v_model.wv.get_vector("love")
print(vector_love)

[ 0.06138203  0.05881222 -0.06370417  0.02444947 -0.20152196 -0.18612292
 -0.15284595  0.4548753  -0.04217871  0.03536078  0.13657516 -0.18520005
 -0.1812647   0.22149836 -0.3038084  -0.23970386  0.07094695 -0.05679139
 -0.05166207 -0.23843557 -0.08530281  0.19564727 -0.07678778  0.03797247
  0.07517307 -0.04826551  0.07379535  0.10396848  0.00738022 -0.22764729
 -0.0456724   0.12937619  0.27785638  0.19387618 -0.13509148  0.20857106
  0.40917322 -0.00387122 -0.1063128  -0.09056759  0.02400028 -0.0800491
  0.13400665  0.08833536 -0.01894405  0.08592905 -0.15905626  0.10259357
  0.14459287 -0.12092585 -0.27919102 -0.04061577  0.11382084  0.31365854
 -0.07409792  0.13976744  0.22791271  0.13209458 -0.01811365  0.09772275
  0.09249583 -0.14871688 -0.16348091 -0.13203284 -0.09834065  0.02714608
  0.16531324  0.26051944 -0.0325964  -0.02894551  0.11621328 -0.06974234
  0.09563565 -0.15276384  0.22071053  0.15996666  0.1589048  -0.04711676
 -0.12555045 -0.03993924 -0.10795183  0.01878959  0.

In [ ]:
# el método `most_similar` también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_love)

[('love', 0.9999999403953552),
 ('babe', 0.9085132479667664),
 ('someone', 0.8886148929595947),
 ('need', 0.8827974200248718),
 ('nothing', 0.8740269541740417),
 ("didn't", 0.8638361096382141),
 ("there's", 0.8526672720909119),
 ('you', 0.8456704616546631),
 ('feed', 0.8445017337799072),
 ('somebody', 0.8362804651260376)]

In [ ]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)

[('babe', 0.9085132479667664),
 ('someone', 0.8886148929595947),
 ('need', 0.8827974200248718),
 ('nothing', 0.8740269541740417),
 ("didn't", 0.8638360500335693),
 ("there's", 0.8526672720909119),
 ('you', 0.8456703424453735),
 ('feed', 0.8445016741752625),
 ('somebody', 0.8362804651260376),
 ('buy', 0.8351694941520691)]

### 5 - Visualizar agrupación de vectores

In [ ]:
from sklearn.decomposition import IncrementalPCA
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions = 2 ):

    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [ ]:
# Graficar los embedddings en 2D
import plotly.graph_objects as go
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS=200
fig = px.scatter(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], text=labels[:MAX_WORDS])
fig.show(renderer="colab") # esto para plotly en colab

In [ ]:
# Graficar los embedddings en 3D

vecs, labels = reduce_dimensions(w2v_model,3)

fig = px.scatter_3d(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], z=vecs[:MAX_WORDS,2],text=labels[:MAX_WORDS])
fig.update_traces(marker_size = 2)
fig.show(renderer="colab") # esto para plotly en colab

In [ ]:
# También se pueden guardar los vectores y labels como tsv para graficar en
# http://projector.tensorflow.org/


vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

### Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con otro artista del dataset Songs.
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)

#### Selección del artista

In [17]:
artista = "bob-marley"

df = pd.read_csv(f'songs_dataset/{artista}.txt', sep='/n', header=None)
display(df.head())
print("\n\nCantidad de documentos:", df.shape[0])


/var/folders/h5/858gwy311wl8z7xr0xvpb78r0000gn/T/ipykernel_28663/2013475072.py:3: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(f'songs_dataset/{artista}.txt', sep='/n', header=None)


,0
0,"""Don't worry about a thing,"
1,'Cause every little thing gonna be all right.
2,"Singin': ""Don't worry about a thing,"
3,"'Cause every little thing gonna be all right!""..."
4,"Smiled with the risin' sun,"




Cantidad de documentos: 2218


#### Preprocesamiento

In [18]:
from gensim.utils import simple_preprocess

# Tokenizar cada línea evitando dependencias pesadas de TensorFlow
sentence_tokens = [
    simple_preprocess(text, deacc=True)
    for text in df[0].dropna().astype(str).tolist()
]

In [19]:
# Se imprime oración y su secuencia correspondiente
print(df.iloc[100][0])
sentence_tokens[100]

And ya chuck it down ya siter,


['and', 'ya', 'chuck', 'it', 'down', 'ya', 'siter']

#### Modelos generador de vectores

In [20]:
from gensim.models.callbacks import CallbackAny2Vec
# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [21]:
# Instaciacion
w2v_model = Word2Vec(min_count=4,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=2,       # cant de palabras antes y desp de la predicha
                     vector_size=200,       # dimensionalidad de los vectores
                     negative=10,    # cantidad de negative samples... 0 es no se usa
                     workers=max(1, multiprocessing.cpu_count() - 1),
                     sg=1)           # modelo 0:CBOW  1:skipgram

In [22]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [23]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 2218


In [24]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 606


#### Entrenamiento de los Embeddings

In [25]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=50,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

Loss after epoch 0: 59438.734375
Loss after epoch 1: 47855.9921875
Loss after epoch 2: 45140.6796875
Loss after epoch 3: 44378.578125
Loss after epoch 4: 42452.171875
Loss after epoch 5: 40723.0625
Loss after epoch 6: 37986.875
Loss after epoch 7: 35805.6875
Loss after epoch 8: 35812.78125
Loss after epoch 9: 32855.96875
Loss after epoch 10: 32116.96875
Loss after epoch 11: 31735.3125
Loss after epoch 12: 30913.96875
Loss after epoch 13: 29618.21875
Loss after epoch 14: 28479.375
Loss after epoch 15: 28254.625
Loss after epoch 16: 27512.1875
Loss after epoch 17: 27008.5
Loss after epoch 18: 25961.5
Loss after epoch 19: 25526.4375
Loss after epoch 20: 25867.9375
Loss after epoch 21: 25015.625
Loss after epoch 22: 24694.5625
Loss after epoch 23: 24604.5
Loss after epoch 24: 23397.5
Loss after epoch 25: 23538.3125
Loss after epoch 26: 23435.25
Loss after epoch 27: 23271.375
Loss after epoch 28: 23224.3125
Loss after epoch 29: 23063.8125
Loss after epoch 30: 22878.0
Loss after epoch 31: 22

(538688, 857350)

#### Evaluación de los embedding

Se evaluan palabras caracteristicas del lenguaje de Bob Marley como "jah", "zion", "rastaman", etc.

El significado de cada palabra fueron obtenidas a partir del siguiente link: [Link a diccionario rastafari](https://www.radionica.rocks/cultura/diccionario-para-entender-la-cultura-del-reggae-y-rastafari)


In [26]:
vocabulario = list(w2v_model.wv.index_to_key)

print(len(vocabulario)) 
print(vocabulario[:50])

606
['you', 'the', 'to', 'and', 'it', 'no', 'we', 'in', 'oh', 'me', 'be', 'love', 'is', 'of', 'yeah', 'all', 'don', 'so', 'your', 'my', 'on', 'for', 'up', 'now', 'down', 'right', 'one', 'can', 're', 'that', 'say', 'this', 'they', 'know', 'do', 'gonna', 'but', 'jah', 'when', 'got', 'de', 'get', 'come', 'let', 'baby', 'there', 'will', 'give', 'll', 'from']


**Palabra: `jah`**

"Expresión clave en la cultura rastafari y el reggae, utilizada como un término cariñoso para referirse a Dios. Su presencia frecuente en las letras de canciones reggae refleja la profunda espiritualidad arraigada a la cultura."

In [21]:
print("Palabras mas similares a 'jah':")
w2v_model.wv.most_similar(positive=["jah"], topn=10)

Palabras mas similares a 'jah':


[('ment', 0.6680909395217896),
 ('protect', 0.6510037183761597),
 ('movement', 0.6283091306686401),
 ('dance', 0.5813745260238647),
 ('send', 0.5714298486709595),
 ('move', 0.5505865812301636),
 ('sufferers', 0.543551504611969),
 ('teachings', 0.5430743098258972),
 ('exodus', 0.5340152978897095),
 ('share', 0.5290785431861877)]

In [22]:
print('Palabras menos similares a "jah":')
w2v_model.wv.most_similar(negative=["jah"], topn=10)

Palabras menos similares a "jah":


[('as', 0.02076398767530918),
 ('talk', 0.018101243302226067),
 ('at', -0.01121883187443018),
 ('them', -0.026404207572340965),
 ('black', -0.0375506766140461),
 ('hungry', -0.04036984592676163),
 ('galang', -0.046791303902864456),
 ('soul', -0.05349563807249069),
 ('gold', -0.05536883324384689),
 ('many', -0.06012966111302376)]

Los resultados obtenidos muestran que el modelo logra capturar adecuadamente el contexto de uso de la palabra “jah” dentro del corpus de canciones.

Las palabras más similares como “protect”, “movement”, “teachings”, “exodus” y “sufferers”, están fuertemente asociadas a temáticas espirituales, sociales y de liberación, que son características del mensaje presente en las letras de Bob Marley. Esto indica que el modelo logra agrupar términos que comparten un mismo contexto conceptual, vinculado a la religión rastafari.

Por otro lado, las palabras menos similares, como “as”, “at” y “them”, corresponden a términos funcionales del lenguaje sin carga semántica relevante, mientras que otras como “gold” o “soul” pertenecen a contextos diferentes dentro del corpus. Esto evidencia que el modelo es capaz de separar adecuadamente palabras según su contexto de uso, lo que ubica a “jah” en una región semántica bien definida.

**Palabra: `zion`**

"Representa un lugar espiritual o utópico cargado de esperanza, libertad y paz. En la música reggae, este término encapsula la aspiración hacia un espacio mejor y libre de opresión."

In [33]:
print('Palabras mas similares a "zion":')
w2v_model.wv.most_similar(positive=["zion"], topn=10)

Palabras mas similares a "zion":


[('mount', 0.9135714173316956),
 ('holy', 0.8180683851242065),
 ('home', 0.7112817764282227),
 ('fly', 0.6970028877258301),
 ('away', 0.6093518137931824),
 ('anytime', 0.6064343452453613),
 ('run', 0.583785355091095),
 ('melodies', 0.5637636184692383),
 ('remain', 0.5568402409553528),
 ('true', 0.5519112348556519)]

In [35]:
print('Palabras menos similares a "zion":')
w2v_model.wv.most_similar(negative=["zion"], topn=10)

Palabras menos similares a "zion":


[('too', 0.011171634308993816),
 ('if', -0.0035557718947529793),
 ('going', -0.027624353766441345),
 ('not', -0.034103792160749435),
 ('wrong', -0.04314947873353958),
 ('you', -0.04613228887319565),
 ('must', -0.07767967134714127),
 ('things', -0.07786095887422562),
 ('me', -0.08425996452569962),
 ('is', -0.0895690768957138)]

Las palabras más similares, como “mount”, “holy”, “home”, “fly” y “away”, reflejan una fuerte asociación con ideas de espiritualidad y busqueda de un lugar sagrado o pertenencia. En particular, términos como “mount” y “holy” refuerzan el carácter religioso del concepto, mientras que “home”, “fly” y “away” sugieren movimiento hacia ese destino ideal.

Por otro lado, las palabras menos similares, como “is”, “me”, “you” y “things”, corresponden a términos funcionales o de uso general que no poseen una carga semántica específica vinculada al concepto de “zion”. Esto indica que el modelo logra ubicar esta palabra en una región semántica bien definida, separándola de términos sin contenido temático relevante.

**Palabra: `rastaman`**

'Rastaman' hace referencia a una persona que forma parte del movimiento rastafari.

In [28]:
print('Palabras mas similares a "rastaman":')
w2v_model.wv.most_similar(positive=["rastaman"], topn=10)

Palabras mas similares a "rastaman":


[('positive', 0.7864192724227905),
 ('sufferers', 0.7794812917709351),
 ('vibration', 0.7374536395072937),
 ('teachings', 0.6918129920959473),
 ('herbsman', 0.6616266369819641),
 ('truth', 0.6606988906860352),
 ('majesty', 0.6546684503555298),
 ('african', 0.6437555551528931),
 ('blood', 0.6269411444664001),
 ('movement', 0.6253232955932617)]

In [29]:
print('Palabras menos similares a "rastaman":')
w2v_model.wv.most_similar(negative=["rastaman"], topn=10)

Palabras menos similares a "rastaman":


[('waiting', 0.04452520236372948),
 ('ain', -0.030074559152126312),
 ('jam', -0.03586288541555405),
 ('too', -0.046210117638111115),
 ('someone', -0.05536185950040817),
 ('running', -0.0567273274064064),
 ('coffin', -0.05874326452612877),
 ('brown', -0.06189991906285286),
 ('there', -0.06436444073915482),
 ('my', -0.06577633321285248)]

Las palabras más similares, como “positive”, “vibration”, “teachings” y “truth”, reflejan claramente los pilares del pensamiento rastafari: espiritualidad, conciencia, identidad cultural y lucha social. Además, términos como “herbsman” y “sufferers” están directamente asociados a las condiciones sociales que el reggae suele abordar, reforzando la coherencia semántica del embedding.

Esto indica que el modelo no solo identifica palabras que coocurren frecuentemente, sino que logra agrupar conceptos que comparten un mismo marco ideológico y cultural, ubicando a “rastaman” en una región semántica bien definida dentro del espacio vectorial.

Por otro lado, las palabras menos similares, como “my”, “there” y “too”, o términos más específicos pero no relacionados como “coffin” o “brown” corresponden a palabras funcionales o a contextos distintos dentro del corpus, lo que evidencia que el modelo separa adecuadamente los usos semánticos relevantes de aquellos que no están vinculados al concepto analizado.

**Palabra: `babylon`**

"Simboliza una sociedad opresora o corrupta, reflejando la crítica y resistencia a sistemas políticos y sociales injustos. Presente en letras de canciones reggae, expresa la lucha contra la adversidad y la búsqueda de justicia y libertad."

In [36]:
print('Palabras mas similares a "babylon":')
w2v_model.wv.most_similar(positive=["babylon"], topn=10)

Palabras mas similares a "babylon":


[('throne', 0.8407189846038818),
 ('leaving', 0.7380030751228333),
 ('gone', 0.6540437936782837),
 ('system', 0.6018829941749573),
 ('rebel', 0.5833534002304077),
 ('feeling', 0.5817123651504517),
 ('ram', 0.5776230692863464),
 ('tomorrow', 0.5755267143249512),
 ('ypocrites', 0.5687455534934998),
 ('su', 0.5350420475006104)]

In [37]:
print('Palabras menos similares a "babylon":')
w2v_model.wv.most_similar(negative=["babylon"], topn=10)

Palabras menos similares a "babylon":


[('give', 0.0025017852894961834),
 ('shed', -0.022582396864891052),
 ('wanna', -0.02264321781694889),
 ('don', -0.024195274338126183),
 ('vain', -0.038175199180841446),
 ('darlin', -0.040080539882183075),
 ('ere', -0.04287504404783249),
 ('love', -0.04305897280573845),
 ('up', -0.04796125367283821),
 ('tears', -0.0515933595597744)]

Entre las palabras más similares aparecen “throne”, “system”, “rebel” y “ypocrites”, que están directamente vinculadas con ideas de poder, dominación y resistencia. Esto es consistente con el uso de “babylon” como símbolo del orden opresivo. Términos como “leaving”, “gone” y “tomorrow” sugieren además la noción de escape o superación de ese sistema, frecuente en las letras.

Las palabras menos similares, como “give”, “love”, “up” y “tears”, son términos generales, lo que indica que el modelo ubica a “babylon” en una región semántica específica, separada del lenguaje cotidiano.

#### Reduccion de la dimensionalidad y gráfico de embeddings

In [27]:
from sklearn.decomposition import IncrementalPCA
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions = 2 ):

    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [29]:
%pip install plotly

  Using cached plotly-6.6.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached narwhals-2.18.1-py3-none-any.whl.metadata (14 kB)
Using cached plotly-6.6.0-py3-none-any.whl (9.9 MB)
Using cached narwhals-2.18.1-py3-none-any.whl (444 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]2m1/2 [plotly]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [32]:
# Graficar los embedddings en 2D
import plotly.graph_objects as go
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS=400
fig = px.scatter(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], text=labels[:MAX_WORDS])
fig.show()

**Grupo 1**

![Grupo de palabras 1](grupo1.png)

La proximidad observada entre palabras como "so", "much", "trouble" y "world" puede explicarse por su coocurrencia en frases recurrentes del corpus, como en la canción *So Much Trouble in the World*. Esto indica que el modelo no solo captura relación semántica, sino también patrones frecuentes, que es común en canciones debido al estribillo o estrofas que se repiten.

Para complementar esta idea, se muestra la cercanía entre 'africa' y 'unite', que proviene de la canción *Africa Unite*

![Grupo de palabras 1b](grupo1b.png)

**Grupo 2**



![Grupo de palabras 1b](grupo2.png)

En esta captura se pueden observar dos grupos:

'buffalo', 'soldier' y 'rasta', que provienen de la canción *Buffalo Soldier* con la frase "Buffalo soldier, dreadlock rasta".

Por otro lado, palabras como 'yo', 'woe', 'woy', 'yoy', 'yoe' provienen de vocalizaciones o coros que, si bien no tienen significado en el lenguaje, el modelo puede reconocerlas por su contexto de aparición.

**Grupo 3**

![Grupo de palabras 3](grupo3.png)

En este grupo se identifican varios subconjuntos semánticos bien definidos. 

Por un lado, términos como "babylon" y "throne" se agrupan en torno a conceptos de poder y sistema, lo que refleja la crítica social característica de las canciones de Bob Marley.

Palabras como "teach", "youths" y "about", "fool" y "blame" conforman un cluster vinculado a la educación y transmisión de mensajes sociales y juicios de valor. Asimismo, "gone" y "today" se relacionan con nociones temporales.

La palabra "simmer" presenta un comportamiento más aislado, posiblemente debido a su baja frecuencia o contexto específico. 

#### Conclusión general

El análisis de palabras similares evidenció que términos con fuerte carga conceptual dentro del dominio como “jah”, “zion”, “rastaman” y “babylon”, se agrupan con otras palabras relacionadas temáticamente, lo que refleja aspectos espirituales, sociales y culturales característicos del reggae y de la filosofia de Bob Marley. Esto indica que los embeddings no solo capturan relaciones lingüísticas generales, sino también particularidades del corpus utilizado.

Además, se observó que el modelo también aprende patrones fraseológicos y estructuras recurrentes de la composición de las canciones. Por ejemplo, la proximidad entre palabras como “so” y “much” se explica por su frecuencia en determinadas canciones y, también, propias del lenguaje, lo que demuestra que los embeddings incorporan tanto relaciones semánticas como sintácticas.

Por otro lado, el análisis también puso en evidencia ciertas limitaciones. En particular, palabras funcionales o de alta frecuencia tienden a ocupar posiciones menos informativas dentro del espacio vectorial, y expresiones fonéticas o propias del lenguaje musical (como vocalizaciones) pueden generar agrupamientos que no responden a un significado semántico claro.

La visualización en dos dimensiones permitió identificar agrupamientos de palabras según su contexto de uso, lo que confirma que el modelo organiza el lenguaje en regiones semánticas diferenciadas. En conjunto, los resultados demuestran que los embeddings constituyen una buena herramienta para modelar el lenguaje natural, lo que permite capturar tanto significado como uso contextual a partir de datos no estructurados.